In [ ]:
! pip install faiss-cpu sentence-transformers tqdm numpy

In [ ]:
import os
import json
import math
import pickle
from typing import List, Dict, Tuple, Optional

import numpy as np
import faiss
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

In [ ]:
import json

In [ ]:
class PMCFAISSRetriever:
    def __init__(
        self,
        model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
        index_type: str = "ivf",
        nlist: int = 4096,
        use_gpu: bool = False,
    ):
        """
        Args:
            model_name: SentenceTransformer model for embeddings.
            index_type: "flat" or "ivf".
            nlist: Number of IVF clusters (used only if index_type == "ivf").
            use_gpu: Whether to move FAISS index to GPU during build/search if possible.
        """
        self.model = SentenceTransformer(model_name)
        self.index_type = index_type.lower()
        self.nlist = nlist
        self.use_gpu = use_gpu

        self.index = None
        self.metadata = []
        self.embedding_dim = self.model.get_sentence_embedding_dimension()

    @staticmethod
    def _normalize(vectors: np.ndarray) -> np.ndarray:
        faiss.normalize_L2(vectors)
        return vectors

    @staticmethod
    def _read_jsonl(path: str) -> List[Dict]:
        with open(path, "r", encoding="utf-8") as f:
            records = json.loads(f.read())
            f.close()
        return records

    @staticmethod
    def _prepare_text(record: Dict) -> str:
        title = record.get("title", "") or ""
        text = record.get("patient", "") or ""
        return f"{title}\n\n{text}".strip()

    def _embed_texts(
        self,
        texts: List[str],
        batch_size: int = 128,
        normalize: bool = True,
        show_progress_bar: bool = True,
    ) -> np.ndarray:
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=show_progress_bar,
            convert_to_numpy=True,
            normalize_embeddings=False,  # we normalize with FAISS for cosine via inner product
        ).astype("float32")

        if normalize:
            embeddings = self._normalize(embeddings)

        return embeddings

    def _create_index(self, embeddings: np.ndarray):
        dim = embeddings.shape[1]

        if self.index_type == "flat":
            # Cosine similarity via normalized embeddings + inner product
            index = faiss.IndexFlatIP(dim)

        elif self.index_type == "ivf":
            quantizer = faiss.IndexFlatIP(dim)
            index = faiss.IndexIVFFlat(quantizer, dim, self.nlist, faiss.METRIC_INNER_PRODUCT)

            # Need training for IVF
            print("Training IVF index...")
            train_size = min(len(embeddings), max(50000, self.nlist * 20))
            train_samples = embeddings[np.random.choice(len(embeddings), train_size, replace=False)]
            index.train(train_samples)

            # Recommended search setting; tune later
            index.nprobe = min(64, self.nlist)

        else:
            raise ValueError("index_type must be either 'flat' or 'ivf'")

        if self.use_gpu:
            try:
                res = faiss.StandardGpuResources()
                index = faiss.index_cpu_to_gpu(res, 0, index)
                print("Using GPU FAISS index.")
            except Exception as e:
                print(f"Could not move FAISS index to GPU, falling back to CPU: {e}")

        return index

    def build_index(
        self,
        input_jsonl: str,
        batch_size: int = 128,
        save_dir: str = "./pmc_faiss_store",
    ):
        """
        Build FAISS index from a JSONL file and save:
            - index.faiss
            - metadata.pkl
            - config.json
        """
        os.makedirs(save_dir, exist_ok=True)

        print("Reading records...")
        records = self._read_jsonl(input_jsonl)

        if len(records) == 0:
            raise ValueError("No records found in input file.")

        print(f"Loaded {len(records)} records.")

        texts = [self._prepare_text(r) for r in records]

        print("Encoding texts...")
        embeddings = self._embed_texts(
            texts=texts,
            batch_size=batch_size,
            normalize=True,
            show_progress_bar=True,
        )

        print(f"Embeddings shape: {embeddings.shape}")

        self.metadata = []
        for i, r in enumerate(records):
            self.metadata.append(
                {
                    "faiss_id": i,
                    "case_id": r.get("patient_id", str(i)),
                    "title": r.get("title", ""),
                    "text": r.get("patient", ""),
                }
            )

        self.index = self._create_index(embeddings)

        print("Adding embeddings to index...")
        self.index.add(embeddings)

        # Move back to CPU before saving if on GPU
        if self.use_gpu:
            try:
                self.index = faiss.index_gpu_to_cpu(self.index)
            except Exception:
                pass

        index_path = os.path.join(save_dir, "index.faiss")
        metadata_path = os.path.join(save_dir, "metadata.pkl")
        config_path = os.path.join(save_dir, "config.json")

        print("Saving index...")
        faiss.write_index(self.index, index_path)

        print("Saving metadata...")
        with open(metadata_path, "wb") as f:
            pickle.dump(self.metadata, f)

        print("Saving config...")
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "model_name": self.model._modules["0"].auto_model.name_or_path
                    if hasattr(self.model, "_modules")
                    else "unknown",
                    "embedding_dim": self.embedding_dim,
                    "index_type": self.index_type,
                    "nlist": self.nlist,
                    "num_records": len(self.metadata),
                },
                f,
                ensure_ascii=False,
                indent=2,
            )

        print(f"Saved FAISS store to: {save_dir}")

    def load(self, save_dir: str = "./pmc_faiss_store"):
        index_path = os.path.join(save_dir, "index.faiss")
        metadata_path = os.path.join(save_dir, "metadata.pkl")
        config_path = os.path.join(save_dir, "config.json")

        if not os.path.exists(index_path):
            raise FileNotFoundError(f"Missing FAISS index: {index_path}")
        if not os.path.exists(metadata_path):
            raise FileNotFoundError(f"Missing metadata: {metadata_path}")
        if not os.path.exists(config_path):
            raise FileNotFoundError(f"Missing config: {config_path}")

        self.index = faiss.read_index(index_path)

        with open(metadata_path, "rb") as f:
            self.metadata = pickle.load(f)

        with open(config_path, "r", encoding="utf-8") as f:
            config = json.load(f)

        if self.use_gpu:
            try:
                res = faiss.StandardGpuResources()
                self.index = faiss.index_cpu_to_gpu(res, 0, self.index)
                print("Loaded index onto GPU.")
            except Exception as e:
                print(f"Could not move loaded index to GPU: {e}")

        print(f"Loaded index with {len(self.metadata)} records.")

    def search(
        self,
        query: str,
        top_k: int = 5,
    ) -> List[Dict]:
        if self.index is None:
            raise ValueError("Index is not loaded or built yet.")

        query_emb = self._embed_texts([query], batch_size=1, normalize=True, show_progress_bar=False)
        scores, indices = self.index.search(query_emb, top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx == -1:
                continue
            item = self.metadata[idx].copy()
            item["score"] = float(score)
            results.append(item)

        return results

In [ ]:
with open('/content/drive/MyDrive/PMC/PMC-Patients.json', 'r') as f:
  data = json.loads(f.read())
  f.close()

In [ ]:
retriever = PMCFAISSRetriever(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        index_type="ivf",   # use "flat" if you want simpler exact search
        nlist=4096,
        use_gpu=True,
    )

retriever.load("/content/drive/MyDrive/pmc_faiss_store")

# 3) Search
query = "brain death determination chinese"
results = retriever.search(query, top_k=30)

print("\nTop results:")
for r in results:
    print("=" * 80)
    print(f"case_id: {r['case_id']}")
    print(f"title: {r['title']}")
    print(f"score: {r['score']:.4f}")
    print(f"text preview: {r['text'][:500]}")


Top results:
case_id: 108582
title: First Organ Donation after Circulatory Death Following Withdrawal of Life-sustaining Treatment in Korea: a Case Report
score: 0.4995
text preview: A 52-year-old male was found in the bathroom where he had collapsed while taking a shower. He was diagnosed with cerebral hemorrhage using computed tomography performed in the emergency room, and extra-ventricular drainage was performed. Subsequently, emergency craniectomy was put forth as an option. However, after being informed about the poor prognosis and high probability of death after surgery, the patient's wife signed a memorandum to forgo the operation. On day 2 of hospitalization, brain 
case_id: 66366
title: Neurological complications and death in children with dengue virus infection: report of two cases
score: 0.4679
text preview: A 14-year-old female was admitted to the hospital on June 5, 2016, at 2:20 am. The patient’s symptoms started 12 h after clinical manifestation in her sister (Case 1).